# AQG Dataset Pipeline — Final Checkpoint

Notebook ini menjalankan pipeline end-to-end dari materi Markdown ke dataset JSONL.
Mulai dari 2 modul dulu (`01-Berkenalan-dengan-python` dan `02-berinteraksi-dengan-data`),
lalu bisa diperluas ke semua modul.

**Alur:**
1. Chunking materi → inspeksi hasil
2. Prompt construction → inspeksi contoh prompt
3. Generate 1 soal sample (test LLM connection)
4. Jalankan pipeline untuk 2 modul
5. Verifikasi output JSONL dengan HuggingFace datasets
6. Jalankan pipeline untuk semua modul (opsional)

> **Catatan Gemini API:**
> - Model: `gemini-2.5-flash` via `langchain-google-genai`
> - API key disimpan di `.env` sebagai `GOOGLE_API_KEY`
> - Jika terjadi error, pipeline **menyimpan data yang sudah berhasil** ke `accumulated.jsonl`
> - Jalankan ulang — pipeline akan melanjutkan dari checkpoint, data lama tidak hilang


In [18]:
import sys, os
from pathlib import Path

# Root project = 2 level di atas src/pipeline/
project_root = Path(os.getcwd())
if project_root.name == 'pipeline':
    project_root = project_root.parent.parent
elif project_root.name == 'src':
    project_root = project_root.parent

os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from dotenv import load_dotenv
load_dotenv()

print(f'Working directory: {os.getcwd()}')
print('Environment loaded.')


Working directory: d:\2-Project\AQG
Environment loaded.


## 1. Chunking — Inspeksi Hasil

In [19]:
from src.dataset.step2.chunker import chunk_markdown, chunk_all_materials

# Test dengan 1 file dulu
chunks = chunk_markdown('dataset_aqg/materi/01-Berkenalan-dengan-python/01-perkenalan-pythn.md')
print(f'Total chunks: {len(chunks)}')
print()
for i, c in enumerate(chunks):
    print(f'[{i}] tokens={c.token_count:3d} | has_code={c.has_code} | heading="{c.section_heading[:50]}"')

Total chunks: 7

[0] tokens=245 | has_code=True | heading="# Pengenalan Python"
[1] tokens=113 | has_code=False | heading="### Python 2.x"
[2] tokens=179 | has_code=False | heading="### Python 3.x"
[3] tokens= 94 | has_code=False | heading="### Python Software Foundation (PSF)"
[4] tokens= 55 | has_code=False | heading="### Python Enhancement Proposals (PEP)"
[5] tokens=309 | has_code=False | heading="### Zen of Python (PEP 20)"
[6] tokens=169 | has_code=False | heading="## Mengapa Python?"


In [20]:

# Lihat isi chunk pertama
print('=== CHUNK 0 ===')
print(chunks[0].text[:500])

=== CHUNK 0 ===
# Pengenalan Python

Halo, calon programmer! Selamat datang pada modul pertama kelas **Memulai Pemrograman dengan Python**. Sebelum Anda mulai membuat program, mari mulai pembelajaran dengan mengenal terlebih dahulu bahasa pemrograman Python. Python adalah bahasa pemrograman multifungsi yang dirilis pada tahun **1991** oleh **Guido van Rossum (GvR)**. Beliau membuat Python sebagai bahasa pemrograman yang mudah dibaca dan dimengerti (*readable*) serta memiliki kemampuan penanganan kesalahan (*exc


In [21]:
# Chunk semua file di modul 01
all_chunks_01 = chunk_all_materials('dataset_aqg/materi/01-Berkenalan-dengan-python')
all_chunks_02 = chunk_all_materials('dataset_aqg/materi/02-berinteraksi-dengan-data')

print(f'Modul 01: {len(all_chunks_01)} chunks')
print(f'Modul 02: {len(all_chunks_02)} chunks')

print(f'Total: {len(all_chunks_01) + len(all_chunks_02)} chunks')

Modul 01: 38 chunks
Modul 02: 34 chunks
Total: 72 chunks


## 2. Prompt Construction — Inspeksi Contoh Prompt

In [22]:
from src.dataset.step2.prompt_constructor import TaskParams, build_prompt
from src.dataset.step2.concept_list import get_concepts_for_module

# Ambil chunk dengan code block
code_chunks = [c for c in all_chunks_01 if c.has_code]
sample_chunk = code_chunks[0] if code_chunks else all_chunks_01[0]

params = TaskParams(concept='Variable dan Assignment', difficulty='medium', question_type='MCQ')
prompt_input = build_prompt(sample_chunk, params)

print('=== INPUT PROMPT ===')
print(prompt_input.input)
print()
print(f'Token count chunk: {sample_chunk.token_count}')
print(f'Has code: {sample_chunk.has_code}')

=== INPUT PROMPT ===
Konteks: # Pengenalan Python

Halo, calon programmer! Selamat datang pada modul pertama kelas **Memulai Pemrograman dengan Python**. Sebelum Anda mulai membuat program, mari mulai pembelajaran dengan mengenal terlebih dahulu bahasa pemrograman Python. Python adalah bahasa pemrograman multifungsi yang dirilis pada tahun **1991** oleh **Guido van Rossum (GvR)**. Beliau membuat Python sebagai bahasa pemrograman yang mudah dibaca dan dimengerti (*readable*) serta memiliki kemampuan penanganan kesalahan (*exception handling*). Berdasarkan tujuan tersebut, Guido van Rossum berhasil menjadikan Python sebagai bahasa pemrograman yang dapat diimplementasikan ke dalam berbagai sektor. Python dapat digunakan untuk membangun website (server-side), analisis data, hingga pembelajaran mesin (*machine learning*). Python memiliki ciri khas tersendiri sebagai salah satu pemrograman populer. Salah satu ciri khas yang paling dikenal adalah Python tidak mewajibkan penggunaan titik koma 

## 3. Test LLM Connection — Generate 1 Soal Sample

In [23]:
from src.dataset.step2.synthetic_generator import _build_llm_client, generate_datapoint

llm = _build_llm_client()
print('LLM client created.')

# Generate 1 soal sample
result = generate_datapoint(prompt_input, llm, max_retries=2)

if result:
    print('\n=== GENERATED DATA POINT ===')
    print(f'TARGET:\n{result.target}')
    print(f'\nMETADATA: {result.metadata}')
else:
    print('GAGAL generate — cek API key dan koneksi')

[DEBUG] model=qwen/qwen3.6-plus:free, api_key=SET
LLM client created.

=== GENERATED DATA POINT ===
TARGET:
Pertanyaan: Berdasarkan teks, simbol apa yang tidak diwajibkan pada akhir setiap baris kode program Python, termasuk saat melakukan assignment variabel? Jawaban benar: Titik koma atau semi colon (`;`). Distraktor: 1) Tanda kurung `()` 2) Tanda kutip `""` 3) Tanda pagar `#` 4) Tanda kurung kurawal `{}`

METADATA: {'difficulty': 'medium', 'question_type': 'MCQ', 'concept': 'Variable dan Assignment', 'misconception_tags': ['butuh_semicolon', 'js_syntax', 'bingung_sintaks_string'], 'source_file': 'dataset_aqg/materi/01-Berkenalan-dengan-python/01-perkenalan-pythn.md', 'section': '# Pengenalan Python', 'source': 'synthetic', 'validated': False}


## 4. Jalankan Pipeline untuk 2 Modul

In [ ]:
from src.pipeline.step2.dataset_pipeline import run_pipeline
import shutil
from pathlib import Path

output_dir = 'dataset_aqg/output_modul'

# JANGAN hapus output_dir jika ingin melanjutkan dari checkpoint!
# Hapus hanya jika ingin mulai dari awal:
# if Path(output_dir).exists():
#     shutil.rmtree(output_dir)
#     print(f'Output lama dihapus: {output_dir}')

accumulated = Path(output_dir) / 'accumulated.jsonl'
if accumulated.exists():
    import json
    count = sum(1 for _ in open(accumulated, encoding='utf-8'))
    print(f'[INFO] Melanjutkan — {count} data point sudah tersimpan di accumulated.jsonl')
else:
    print('[INFO] Mulai dari awal.')

print('Menjalankan pipeline untuk modul 01...')

In [ ]:
# Modul 01
run_pipeline(
    materi_dir='dataset_aqg/materi',
    output_dir=output_dir,
    section_filter='01-Berkenalan-dengan-python',
    difficulties=['easy', 'medium', 'hard'],
    question_types=['MCQ'],
    max_chunks_per_section=30,  # 10 chunk → 30 prompt total
)

In [ ]:
# Modul 02
run_pipeline(
    materi_dir='dataset_aqg/materi',
    output_dir=output_dir,
    section_filter='02-berinteraksi-dengan-data',
    difficulties=['easy', 'medium', 'hard'],
    question_types=['MCQ', 'Code Completion'],
    max_chunks_per_section=30,
)


In [ ]:
# Modul 03
run_pipeline(
    materi_dir='dataset_aqg/materi',
    output_dir=output_dir,
    section_filter='03-ekspresi',
    difficulties=['easy', 'medium', 'hard'],
    question_types=['MCQ', 'Code Completion'],
    # max_chunks_per_section=30,
)


In [ ]:
# Modul 04
run_pipeline(
    materi_dir='dataset_aqg/materi',
    output_dir=output_dir,
    section_filter='04-aksi-sekuensial',
    difficulties=['easy', 'medium', 'hard'],
    question_types=['MCQ', 'Code Completion'],
    # max_chunks_per_section=30,
)


In [ ]:
# Modul 05
run_pipeline(
    materi_dir='dataset_aqg/materi',
    output_dir=output_dir,
    section_filter='05-control-flow',
    difficulties=['easy', 'medium', 'hard'],
    question_types=['MCQ', 'Code Completion'],
    max_chunks_per_section=20,
)


In [ ]:
# Modul 06
run_pipeline(
    materi_dir='dataset_aqg/materi',
    output_dir=output_dir,
    section_filter='06-array',
    difficulties=['easy', 'medium', 'hard'],
    question_types=['MCQ', 'Code Completion'],
    # max_chunks_per_section=20,
)


In [ ]:
# Modul 07
run_pipeline(
    materi_dir='dataset_aqg/materi',
    output_dir=output_dir,
    section_filter='07-matriks',
    difficulties=['easy', 'medium', 'hard'],
    question_types=['MCQ', 'Code Completion'],
    # max_chunks_per_section=20,
)


In [ ]:
# Modul 08
run_pipeline(
    materi_dir='dataset_aqg/materi',
    output_dir=output_dir,
    section_filter='08-subprogram',
    difficulties=['easy', 'medium', 'hard'],
    question_types=['MCQ', 'Code Completion'],
    max_chunks_per_section=20,
)


In [ ]:
# Modul 09
run_pipeline(
    materi_dir='dataset_aqg/materi',
    output_dir=output_dir,
    section_filter='09-oop',
    difficulties=['easy', 'medium', 'hard'],
    question_types=['MCQ', 'Code Completion'],
    # max_chunks_per_section=20,
)


In [ ]:
# Modul 10
run_pipeline(
    materi_dir='dataset_aqg/materi',
    output_dir=output_dir,
    section_filter='10-style-guide',
    difficulties=['easy', 'medium', 'hard'],
    question_types=['MCQ', 'Code Completion'],
    max_chunks_per_section=20,
)


## 5. Verifikasi Output JSONL

In [ ]:
import json
from pathlib import Path

# Cek file yang dihasilkan
output_path = Path(output_dir)
for f in sorted(output_path.iterdir()):
    size = f.stat().st_size
    print(f'{f.name:30s} {size:8d} bytes')

In [ ]:
# Baca dataset_info.json
with open(output_path / 'dataset_info.json', encoding='utf-8') as f:
    info = json.load(f)

print('=== DATASET INFO ===')
print(json.dumps(info, indent=2, ensure_ascii=False))

In [ ]:
# Inspeksi 3 contoh dari train.jsonl
train_file = output_path / 'train.jsonl'
samples = []
with open(train_file, encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i >= 3:
            break
        samples.append(json.loads(line))

for i, s in enumerate(samples):
    print(f'\n=== SAMPLE {i+1} ===')
    print(f'CONCEPT: {s["metadata"]["concept"]}')
    print(f'DIFFICULTY: {s["metadata"]["difficulty"]}')
    print(f'TARGET:\n{s["target"]}')

In [ ]:
# Verifikasi dengan HuggingFace datasets
try:
    from datasets import load_dataset
    ds = load_dataset('json', data_files={
        'train': str(output_path / 'train.jsonl'),
        'validation': str(output_path / 'validation.jsonl'),
        'test': str(output_path / 'test.jsonl'),
    })
    print('=== HUGGINGFACE DATASET ===')
    print(ds)
    print(f'\nContoh dari train split:')
    print(ds['train'][0])
except ImportError:
    print('datasets library tidak tersedia — skip HuggingFace verification')

## 6. Evaluais hasil dataset

Uncomment cell di bawah untuk menjalankan pipeline pada semua 11 modul.
Estimasi waktu: ~30-60 menit tergantung rate limit OpenRouter.

In [ ]:
## 8. Evaluasi Dataset per Modul

import json, os
from pathlib import Path
from collections import Counter
import matplotlib.pyplot as plt

# Resolve absolute path
_candidates = [
    Path('dataset_aqg/output_modul'),
    Path('../../dataset_aqg/output_modul'),
    Path(os.getcwd()) / 'dataset_aqg' / 'output_modul',
]
OUTPUT_ROOT = next((p.resolve() for p in _candidates if p.exists()), None)
assert OUTPUT_ROOT, "Tidak bisa menemukan output_modul — jalankan cell setup dulu"
print(f'OUTPUT_ROOT: {OUTPUT_ROOT}')

def load_accumulated(folder: Path) -> list[dict]:
    f = folder / 'accumulated.jsonl'
    if not f.exists():
        return []
    with open(f, encoding='utf-8') as fh:
        return [json.loads(l) for l in fh if l.strip()]

def evaluate_module(name: str, data: list[dict]) -> dict:
    n = len(data)
    if n == 0:
        return {'name': name, 'total': 0}
    diff      = Counter(d['metadata'].get('difficulty', 'unknown') for d in data)
    qtype     = Counter(d['metadata'].get('question_type', 'unknown') for d in data)
    validated = sum(1 for d in data if d['metadata'].get('validated') is True)
    has_tags  = sum(1 for d in data if d['metadata'].get('misconception_tags'))
    concepts  = Counter(d['metadata'].get('concept', 'unknown') for d in data)
    diff_vals = [diff.get(k, 0) for k in ['easy', 'medium', 'hard']]
    nonzero   = [v for v in diff_vals if v > 0]
    imbalance = round(max(nonzero) / min(nonzero), 2) if len(nonzero) > 1 else float('inf')
    return {
        'name': name, 'total': n,
        'difficulty': dict(diff), 'question_type': dict(qtype),
        'validated': validated, 'has_tags': has_tags,
        'concepts': concepts, 'imbalance_ratio': imbalance,
    }

# Load semua subfolder
modules = {}
for folder in sorted(OUTPUT_ROOT.iterdir()):
    if folder.is_dir():
        data = load_accumulated(folder)
        if data:
            modules[folder.name] = evaluate_module(folder.name, data)

if not modules:
    print('[WARNING] Tidak ada accumulated.jsonl di subfolder manapun')
    print('Subfolder:', [f.name for f in OUTPUT_ROOT.iterdir() if f.is_dir()])
else:
    # Ringkasan teks
    print(f"\n{'Modul':<45} {'Total':>6} {'Easy':>6} {'Med':>5} {'Hard':>5} {'Ratio':>7} {'Val%':>6} {'Tags%':>6}")
    print('-' * 95)
    for name, ev in modules.items():
        n = ev['total']
        d = ev['difficulty']
        val_pct  = round(ev['validated'] / n * 100) if n else 0
        tags_pct = round(ev['has_tags']  / n * 100) if n else 0
        status = '✓ OK' if ev['imbalance_ratio'] <= 2.0 else '⚠ IMBALANCED'
        print(f"{name:<45} {n:>6} {d.get('easy',0):>6} {d.get('medium',0):>5} {d.get('hard',0):>5} {ev['imbalance_ratio']:>7} {val_pct:>5}% {tags_pct:>5}%  {status}")

    # Visualisasi
    n_mods = len(modules)
    fig, axes = plt.subplots(n_mods, 3, figsize=(15, 4 * n_mods))
    if n_mods == 1:
        axes = [axes]

    for row, (name, ev) in enumerate(modules.items()):
        short = name[:35]

        ax = axes[row][0]
        keys = ['easy', 'medium', 'hard']
        vals = [ev['difficulty'].get(k, 0) for k in keys]
        bars = ax.bar(keys, vals, color=['#4CAF50', '#2196F3', '#F44336'])
        ax.set_title(f'{short}\nDifficulty  (ratio={ev["imbalance_ratio"]}x)')
        ax.set_ylabel('Jumlah soal')
        for b, v in zip(bars, vals):
            if v > 0:
                ax.text(b.get_x() + b.get_width()/2, v + 0.1, str(v), ha='center', fontsize=9)

        ax = axes[row][1]
        qt = ev['question_type']
        ax.bar(list(qt.keys()), list(qt.values()), color='#9C27B0')
        ax.set_title(f'{short}\nQuestion Type')
        ax.set_ylabel('Jumlah soal')

        ax = axes[row][2]
        top10 = ev['concepts'].most_common(10)
        if top10:
            labels, counts = zip(*top10)
            ax.barh(list(labels)[::-1], list(counts)[::-1], color='#FF9800')
            ax.set_title(f'{short}\nTop 10 Concepts')
            ax.set_xlabel('Jumlah soal')

    plt.tight_layout()
    out_png = OUTPUT_ROOT / 'evaluasi_distribusi.png'
    plt.savefig(str(out_png), dpi=120, bbox_inches='tight')
    plt.show()
    print(f'\nGrafik disimpan ke {out_png}')


## 7. Dry Run — Test Pipeline Tanpa LLM

In [ ]:
# Dry run untuk verifikasi pipeline tanpa memanggil LLM
import shutil

dry_output = 'dataset_aqg/output_dryrun'
if Path(dry_output).exists():
    shutil.rmtree(dry_output)

run_pipeline(
    materi_dir='dataset_aqg/materi',
    output_dir=dry_output,
    max_per_chunk=1,
    section_filter='01-Berkenalan-dengan-python',
    dry_run=True,
)

# Verifikasi output dry run
dry_path = Path(dry_output)
with open(dry_path / 'dataset_info.json', encoding='utf-8') as f:
    dry_info = json.load(f)
print(f'Dry run total: {dry_info["total"]} data points')
print(f'Splits: {dry_info["splits"]}')